# Model performance across one or many runs

Generalizes **section 12** of `10_vanilla_dt.ipynb` to plot the four DT performance
figures for a **list of model run directories** — training curves, acquisition sanity,
probe performance, and performance profiles.

- **One model** -> output reproduces section 12 (faceted by subject and temperature).
- **Many models** -> *overlay-pooled* comparison: all models share axes, one color per
  model, pooling across each model's subjects and seeds. The two temperature-colored
  plots (probe performance, performance profiles) fix temperature to `OVERLAY_TEMP`.

Self-contained: needs only `pandas` / `numpy` / `matplotlib` (no `corner_maze_rl` import).

## 1. Configuration

Set `MODELS` to the run directory name(s) you want to plot.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Repo root: this notebook lives in <repo>/notebooks/
REPO_ROOT = Path.cwd().parent
RUNS_DIR  = REPO_ROOT / 'data' / 'runs'

# ── Models to plot ───────────────────────────────────────────────────────────
# MODELS accepts any of:
#   * a list of run-directory NAMES (resolved under RUNS_DIR)
#   * a list of absolute paths
#   * a dict {legend_label: name-or-path}
# One model   -> reproduces section 12 of 10_vanilla_dt.ipynb (faceted by subject/temp).
# Many models -> overlay-pooled comparison (one color per model, subjects+seeds pooled).
MODELS = [
    'dt-vanilla-pi-cm-3-4-7-15-18-pose_visual-epoch-50-kill-5-test-batch-512_20260531_070527',
]

OVERLAY_TEMP              = 'det'   # temperature fixed in multi-model probe/profile plots
ACQ_PASS_COMPLETED_TRIALS = 24      # acquisition pass threshold (NOT stored in run_config.json)
SAVE_DIR                  = None    # None = show only; set to a Path to also write PNGs


## 2. Resolve models + pooled loaders

`resolve_models` turns `MODELS` into an ordered `{label: run_root}` map (short, unique
legend labels). `load_concat` globs each run's per-seed parquets and tags rows with a
`model` column, dropping (with a printed note) any model missing a given file.

In [ ]:
# ── Resolve MODELS -> ordered {label: run_root Path}; pooled-load helpers ──
_TS_RE = re.compile(r'_\d{8}_\d{6}$')   # trailing _YYYYMMDD_HHMMSS


def _short_label(basename: str) -> str:
    """Concise legend label: drop the leading 'dt-vanilla-' and trailing timestamp."""
    s = _TS_RE.sub('', basename)
    if s.startswith('dt-vanilla-'):
        s = s[len('dt-vanilla-'):]
    return s


def resolve_models(models, runs_dir) -> dict[str, Path]:
    """Return an ordered {label: run_root Path}.

    `models` may be a {label: path} dict (labels used verbatim) or a list of
    names/paths (labels auto-derived, short + unique). Names resolve under
    runs_dir; absolute paths are used as-is. On a post-strip label collision
    (same name, different timestamp) we fall back to the full basename.
    """
    runs_dir = Path(runs_dir)

    def _path(p):
        p = Path(p)
        return p if p.is_absolute() else runs_dir / p

    if isinstance(models, dict):
        return {lbl: _path(p) for lbl, p in models.items()}

    paths  = [_path(m) for m in models]
    shorts = [_short_label(p.name) for p in paths]
    counts = {}
    for s in shorts:
        counts[s] = counts.get(s, 0) + 1
    out: dict[str, Path] = {}
    for p, s in zip(paths, shorts):
        label = s if counts[s] == 1 else p.name          # full basename on collision
        base, k = label, 2
        while label in out:                              # guard against any residual dup
            label = f'{base}#{k}'; k += 1
        out[label] = p
    return out


def load_concat(model_paths, rel_glob) -> pd.DataFrame:
    """Glob `rel_glob` under each run root, concat, tag each row with `model`.

    A model missing the file is dropped with a printed note (never silent). If no
    model has the file, an empty DataFrame is returned so the plot can skip itself.
    """
    frames = []
    for label, root in model_paths.items():
        files = sorted(Path(root).glob(rel_glob))
        if not files:
            print(f'  note: no {rel_glob} under model "{label}" ({root}) — dropping from this plot')
            continue
        df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
        df['model'] = label
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def read_acq_trials(run_root, fallback: int = 32) -> int:
    """inference.acq_trials from any seed's run_config.json under this run root."""
    for cfg in sorted(Path(run_root).glob('*/seed_*/run_config.json')):
        try:
            with open(cfg) as f:
                return int(json.load(f).get('inference', {}).get('acq_trials', fallback))
        except Exception:
            continue
    return fallback


model_paths          = resolve_models(MODELS, RUNS_DIR)
ACQ_TRIALS_BY_MODEL  = {lbl: read_acq_trials(p) for lbl, p in model_paths.items()}
ACQ_INFERENCE_TRIALS = next(iter(ACQ_TRIALS_BY_MODEL.values()), 32)

print(f'{len(model_paths)} model(s) resolved:')
for lbl, p in model_paths.items():
    status = 'ok' if Path(p).exists() else 'MISSING'
    print(f'  [{status:>7}] {lbl}  ->  {p}  (acq_trials={ACQ_TRIALS_BY_MODEL[lbl]})')


## 3. Stats + probe-bucketing helpers

Copied verbatim from section 12 (cell 26) of `10_vanilla_dt.ipynb`.

In [ ]:
# ── Stats + probe-bucketing helpers (verbatim from section 12 of 10_vanilla_dt.ipynb) ──
# numpy replacements for rliable (avoids pandas-3 / arch / statsmodels conflict)
def _iqm(arr: np.ndarray) -> float:
    """Interquartile mean: mean of values within [Q1, Q3]."""
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return float('nan')
    if len(arr) == 1:
        return float(arr[0])
    q1, q3 = np.quantile(arr, [0.25, 0.75])
    keep = arr[(arr >= q1) & (arr <= q3)]
    return float(keep.mean()) if len(keep) else float(arr.mean())


def _iqm_ci(per_seed_scores: np.ndarray, reps: int = 2000,
            seed: int = 0) -> tuple[float, float, float]:
    """IQM + 95% percentile-bootstrap CI from a per-seed score array."""
    arr = np.asarray(per_seed_scores, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return float('nan'), float('nan'), float('nan')
    if len(arr) == 1:
        v = float(arr[0])
        return v, v, v
    rng     = np.random.default_rng(seed)
    n       = len(arr)
    samples = rng.integers(0, n, size=(reps, n))
    boots   = np.array([_iqm(arr[idx]) for idx in samples])
    lo, hi  = np.percentile(boots, [2.5, 97.5])
    return _iqm(arr), float(lo), float(hi)


def _perf_profile(scores_matrix: np.ndarray, tau: np.ndarray,
                  reps: int = 2000, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Performance profile P(score >= tau) over (seeds × tasks) cells.

    Returns (point, ci) where:
      point: shape (len(tau),)
      ci:    shape (2, len(tau)) — [lo, hi] at 95%
    """
    arr = np.asarray(scores_matrix, dtype=float)
    arr = np.nan_to_num(arr, nan=np.nanmean(arr) if np.any(~np.isnan(arr)) else 0.0)
    flat = arr.reshape(-1)
    point = np.array([(flat >= t).mean() for t in tau])

    rng = np.random.default_rng(seed)
    n   = len(flat)
    boots = np.empty((reps, len(tau)))
    for r in range(reps):
        idx        = rng.integers(0, n, size=n)
        resampled  = flat[idx]
        boots[r]   = [(resampled >= t).mean() for t in tau]
    ci = np.percentile(boots, [2.5, 97.5], axis=0)
    return point, ci


def _per_seed_probe_scores(run_root: Path) -> pd.DataFrame:
    paths = sorted(run_root.glob('*/seed_*/probes.parquet'))
    if not paths:
        return pd.DataFrame()
    all_probes = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    per_seed = (all_probes.groupby(['unit_label', 'probe_session_type', 'trial_tag', 'temp_label', 'seed'])
                .agg(n_trials=('dt_completed', 'count'),
                     n_completed=('dt_completed', 'sum'))
                .reset_index())
    per_seed['rate'] = per_seed['n_completed'] / per_seed['n_trials']
    return per_seed


# Probe-session bucketing: which probe paradigms get a training-vs-probe split
# vs a single overall score. Match by substring on probe_session_type so this
# works regardless of the training-group prefix ("PI+VC f2 novel route", etc.).
_PROBE_SPLIT_KEYWORDS = ('novel route', 'reversal')
_TRAINING_TAGS        = {'trained', 'probe_trained'}
_PROBE_MANIP_TAGS     = {'novel', 'reversal'}


def _probe_session_group(probe_type: str, tag: str) -> str:
    """Classify a probe trial into 'training', 'probe', or 'overall'.

    For probes in _PROBE_SPLIT_KEYWORDS, training-style tags become the in-session
    baseline and manipulation tags become the probe bar. Other probes pool every
    trial into 'overall' so the bar reflects the full session score.
    """
    if any(kw in probe_type for kw in _PROBE_SPLIT_KEYWORDS):
        if tag in _TRAINING_TAGS:    return 'training'
        if tag in _PROBE_MANIP_TAGS: return 'probe'
        return 'other'
    return 'overall'


## 4. Plot functions

Each `plot_*` dispatches on the number of models: the single-model path runs the verbatim
section-12 body; the multi-model path runs the overlay-pooled branch.

In [ ]:
# ── Plot dispatchers ─────────────────────────────────────────────────────────
# Each public plot_* takes the {label: run_root} dict. With exactly one model it
# runs the verbatim section-12 body (faceted by subject/temperature). With more
# than one it runs the overlay-pooled branch: `model` is the color dimension and
# subjects + seeds are pooled. For the two temperature-colored plots (probe perf,
# performance profiles) the multi-model branch fixes temperature to OVERLAY_TEMP.


def _save_show(fig, name, show, save_dir):
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        out = save_dir / name
        fig.savefig(out, dpi=120, bbox_inches='tight')
        print(f'  wrote {out}')
    if show:
        plt.show()
    else:
        plt.close(fig)


def _model_color_map(model_paths):
    """Stable {label: color} over the FULL model order (so a model keeps its
    color across plots even if it is dropped from some of them)."""
    labels = list(model_paths)
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(labels), 1)))
    return dict(zip(labels, colors))


# ── single-model bodies (section-12 behaviour; FIGURES_DIR -> save_dir) ──────
def _plot_training_curves_single(run_root, *, show=True, save_dir=None):
    run_root = Path(run_root)
    curves_paths = sorted(run_root.glob('*/seed_*/curves.parquet'))
    if not curves_paths:
        print('no curves.parquet — skipping plot_training_curves')
        return
    all_curves = pd.concat([pd.read_parquet(p) for p in curves_paths], ignore_index=True)

    units = sorted(all_curves['unit_label'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(units), 1)))

    for unit, color in zip(units, colors):
        sub = all_curves[all_curves['unit_label'] == unit]
        for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train loss', 'Val loss']):
            stats = (sub.groupby('epoch')[col]
                     .agg(median='median',
                          p25=lambda x: x.quantile(0.25),
                          p75=lambda x: x.quantile(0.75))
                     .reset_index())
            ax.plot(stats['epoch'], stats['median'], color=color, label=unit, lw=2)
            ax.fill_between(stats['epoch'], stats['p25'], stats['p75'], alpha=0.25, color=color)
            ax.set_title(title); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
            ax.grid(True, alpha=0.3)
    axes[0].legend(loc='upper right', fontsize=9)
    fig.suptitle(f'Training curves (median + p25/p75 across seeds, n={all_curves["seed"].nunique()})')
    fig.tight_layout()
    _save_show(fig, 'training_curves.png', show, save_dir)


def _plot_acq_sanity_single(run_root, *, show=True, save_dir=None,
                            acq_inference_trials=32, acq_pass_trials=24):
    run_root = Path(run_root)
    acq_paths = sorted(run_root.glob('*/seed_*/acq_inference.parquet'))
    if not acq_paths:
        print('no acq_inference.parquet — skipping plot_acq_sanity')
        return
    all_acq = pd.concat([pd.read_parquet(p) for p in acq_paths], ignore_index=True)
    per_seed = (all_acq.groupby(['unit_label', 'temp_label', 'seed'])
                .agg(n_correct=('dt_correct', 'sum'),
                     n_trials=('dt_correct', 'count'))
                .reset_index())
    per_seed['rate'] = per_seed['n_correct'] / per_seed['n_trials']

    units = sorted(per_seed['unit_label'].unique())
    temps = sorted(per_seed['temp_label'].unique())
    n_units = len(units)
    fig, axes = plt.subplots(1, n_units, figsize=(max(4 * n_units, 5), 5), sharey=True)
    if n_units == 1: axes = [axes]
    pass_rate = acq_pass_trials / acq_inference_trials
    rng = np.random.RandomState(0)
    for ax, unit in zip(axes, units):
        sub = per_seed[per_seed['unit_label'] == unit]
        for i, temp in enumerate(temps):
            rates = sub[sub['temp_label'] == temp]['rate'].values
            jitter = rng.uniform(-0.1, 0.1, size=len(rates))
            ax.scatter(np.full(len(rates), i) + jitter, rates,
                       s=80, alpha=0.7, color='steelblue', edgecolors='black', zorder=3)
            if len(rates):
                ax.plot([i - 0.25, i + 0.25], [rates.mean(), rates.mean()],
                        color='black', lw=2, zorder=4)
        ax.axhline(pass_rate, color='red', linestyle='--', alpha=0.7, zorder=2,
                   label=f'pass threshold ({acq_pass_trials}/{acq_inference_trials} = {pass_rate:.0%})')
        ax.set_xticks(range(len(temps))); ax.set_xticklabels(temps)
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.set_xlabel('inference temperature')
        ax.set_title(unit)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel(f'per-seed % correct (of {acq_inference_trials} trials)')
    axes[0].legend(loc='lower right', fontsize=9)
    fig.suptitle('Acquisition sanity check — strict-correct rate (dots = seeds; bar = mean)')
    fig.tight_layout()
    _save_show(fig, 'acq_sanity.png', show, save_dir)


def _plot_probe_performance_single(run_root, *, show=True, save_dir=None):
    run_root = Path(run_root)
    paths = sorted(run_root.glob('*/seed_*/probes.parquet'))
    if not paths:
        print('no probes.parquet — skipping plot_probe_performance')
        return
    all_probes = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    all_probes['group'] = [
        _probe_session_group(pt, tg)
        for pt, tg in zip(all_probes['probe_session_type'], all_probes['trial_tag'])
    ]
    all_probes = all_probes[all_probes['group'] != 'other']

    units   = sorted(all_probes['unit_label'].unique())
    probes  = sorted(all_probes['probe_session_type'].unique())
    temps   = sorted(all_probes['temp_label'].unique())
    palette = dict(zip(temps, plt.cm.tab10(np.linspace(0, 1, max(len(temps), 1)))))

    n_units, n_probes = len(units), len(probes)
    fig, axes = plt.subplots(n_units, n_probes,
                             figsize=(max(4 * n_probes, 6), max(3.8 * n_units, 4)),
                             sharey=True, squeeze=False)

    rng = np.random.RandomState(0)
    for ui, unit in enumerate(units):
        for pi, probe in enumerate(probes):
            ax   = axes[ui, pi]
            cell = all_probes[(all_probes['unit_label'] == unit) &
                              (all_probes['probe_session_type'] == probe)]
            split  = any(kw in probe for kw in _PROBE_SPLIT_KEYWORDS)
            wanted = ['training', 'probe'] if split else ['overall']
            groups_here = [g for g in wanted if (cell['group'] == g).any()]
            if not groups_here:
                ax.set_visible(False)
                continue
            x       = np.arange(len(groups_here))
            n_temps = len(temps)
            width   = 0.8 / max(n_temps, 1)

            for ti, temp in enumerate(temps):
                offset = (ti - (n_temps - 1) / 2) * width
                labeled_temp = False
                for gi, group in enumerate(groups_here):
                    sub = cell[(cell['temp_label'] == temp) & (cell['group'] == group)]
                    if sub.empty:
                        continue
                    pooled = float(sub['dt_correct'].sum()) / float(len(sub))
                    ax.bar(x[gi] + offset, pooled, width,
                           color=palette[temp], edgecolor='black', alpha=0.85, zorder=2,
                           label=(temp if not labeled_temp else None))
                    labeled_temp = True
                    per_seed = (sub.groupby('seed')['dt_correct']
                                .agg(['sum', 'count']).reset_index())
                    per_seed['rate'] = per_seed['sum'] / per_seed['count']
                    jitter = rng.uniform(-width / 4, width / 4, size=len(per_seed))
                    ax.scatter(np.full(len(per_seed), x[gi] + offset) + jitter,
                               per_seed['rate'].values,
                               s=22, alpha=0.75, color='black', edgecolors='white',
                               linewidth=0.5, zorder=3)
            ax.set_xticks(x); ax.set_xticklabels(groups_here, fontsize=9)
            ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3, axis='y')
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
            if ui == 0:
                ax.set_title(probe, fontsize=10)
            if pi == 0:
                ax.set_ylabel(f'{unit}\n% correct (of total trials)')
            if ui == 0 and pi == n_probes - 1:
                ax.legend(loc='upper right', fontsize=8, title='temp')
    fig.suptitle('Probe performance (DT) — pooled correct/total per session group; dots = per-seed')
    fig.tight_layout()
    _save_show(fig, 'probe_performance.png', show, save_dir)


def _plot_performance_profiles_single(run_root, *, show=True, save_dir=None, reps=2000):
    run_root = Path(run_root)
    per_seed = _per_seed_probe_scores(run_root)
    if per_seed.empty:
        print('no probes.parquet — skipping plot_performance_profiles')
        return

    units = sorted(per_seed['unit_label'].unique())
    temps = sorted(per_seed['temp_label'].unique())
    tau   = np.linspace(0.0, 1.0, 51)

    fig, axes = plt.subplots(1, len(units), figsize=(max(5 * len(units), 5), 4), sharey=True, squeeze=False)
    axes = axes[0]
    palette = dict(zip(temps, plt.cm.tab10(np.linspace(0, 1, len(temps)))))
    for ax, unit in zip(axes, units):
        sub   = per_seed[per_seed['unit_label'] == unit]
        tasks = sorted({(p, t) for p, t in zip(sub['probe_session_type'], sub['trial_tag'])})
        seeds = sorted(sub['seed'].unique())
        for temp in temps:
            arr = np.full((len(seeds), len(tasks)), np.nan)
            for r, sd in enumerate(seeds):
                for c, (p, t) in enumerate(tasks):
                    cell = sub[(sub['temp_label'] == temp) & (sub['seed'] == sd)
                               & (sub['probe_session_type'] == p) & (sub['trial_tag'] == t)]
                    if len(cell):
                        arr[r, c] = float(cell['rate'].iloc[0])
            pt, ci = _perf_profile(arr, tau, reps=reps)
            color = palette[temp]
            ax.plot(tau, pt, color=color, label=temp, lw=2)
            ax.fill_between(tau, ci[0], ci[1], color=color, alpha=0.25)
        ax.set_title(unit); ax.grid(True, alpha=0.3)
        ax.set_xlabel(r'Success rate threshold $\tau$')
        ax.set_ylabel(r'P(score $\geq\tau$)')
        ax.set_ylim(0, 1.02); ax.set_xlim(0, 1)
        ax.legend(loc='upper right', fontsize=9)

    fig.suptitle(f'Performance profiles across (probe x tag) tasks (bootstrap CI, reps={reps})')
    fig.tight_layout()
    _save_show(fig, 'performance_profiles.png', show, save_dir)


# ── multi-model overlay branches (pool subjects + seeds; color = model) ──────
def _plot_training_curves_overlay(model_paths, *, show=True, save_dir=None):
    df = load_concat(model_paths, '*/seed_*/curves.parquet')
    if df.empty:
        print('no curves.parquet — skipping plot_training_curves')
        return
    cmap   = _model_color_map(model_paths)
    models = [m for m in model_paths if m in set(df['model'])]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for model in models:
        color = cmap[model]
        sub = df[df['model'] == model]
        for ax, col, title in zip(axes, ['train_loss', 'val_loss'], ['Train loss', 'Val loss']):
            stats = (sub.groupby('epoch')[col]
                     .agg(median='median',
                          p25=lambda x: x.quantile(0.25),
                          p75=lambda x: x.quantile(0.75))
                     .reset_index())
            ax.plot(stats['epoch'], stats['median'], color=color, label=model, lw=2)
            ax.fill_between(stats['epoch'], stats['p25'], stats['p75'], alpha=0.25, color=color)
            ax.set_title(title); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
            ax.grid(True, alpha=0.3)
    axes[0].legend(loc='upper right', fontsize=9)
    fig.suptitle('Training curves by model (median + p25/p75 pooled over subjects×seeds)')
    fig.tight_layout()
    _save_show(fig, 'training_curves.png', show, save_dir)


def _plot_acq_sanity_overlay(model_paths, *, show=True, save_dir=None,
                             acq_inference_trials=32, acq_pass_trials=24):
    df = load_concat(model_paths, '*/seed_*/acq_inference.parquet')
    if df.empty:
        print('no acq_inference.parquet — skipping plot_acq_sanity')
        return
    per_unit = (df.groupby(['model', 'temp_label', 'unit_label', 'seed'])
                .agg(n_correct=('dt_correct', 'sum'),
                     n_trials=('dt_correct', 'count'))
                .reset_index())
    per_unit['rate'] = per_unit['n_correct'] / per_unit['n_trials']

    cmap    = _model_color_map(model_paths)
    models  = [m for m in model_paths if m in set(per_unit['model'])]
    temps   = sorted(per_unit['temp_label'].unique())
    n_temps = len(temps)
    fig, axes = plt.subplots(1, n_temps, figsize=(max(4 * n_temps, 5), 5), sharey=True)
    if n_temps == 1: axes = [axes]
    pass_rate = acq_pass_trials / acq_inference_trials
    rng = np.random.RandomState(0)
    for ax, temp in zip(axes, temps):
        sub = per_unit[per_unit['temp_label'] == temp]
        for i, model in enumerate(models):
            rates = sub[sub['model'] == model]['rate'].values
            jitter = rng.uniform(-0.1, 0.1, size=len(rates))
            ax.scatter(np.full(len(rates), i) + jitter, rates,
                       s=80, alpha=0.7, color=cmap[model], edgecolors='black', zorder=3)
            if len(rates):
                ax.plot([i - 0.25, i + 0.25], [rates.mean(), rates.mean()],
                        color='black', lw=2, zorder=4)
        ax.axhline(pass_rate, color='red', linestyle='--', alpha=0.7, zorder=2,
                   label=f'pass threshold ({acq_pass_trials}/{acq_inference_trials} = {pass_rate:.0%})')
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=30, ha='right', fontsize=8)
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.set_title(f'temp = {temp}')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('per-(subject,seed) % correct')
    axes[0].legend(loc='lower right', fontsize=9)
    fig.suptitle('Acquisition sanity by model (dots = subject×seed; bar = mean)')
    fig.tight_layout()
    _save_show(fig, 'acq_sanity.png', show, save_dir)


def _plot_probe_performance_overlay(model_paths, *, overlay_temp='det', show=True, save_dir=None):
    df = load_concat(model_paths, '*/seed_*/probes.parquet')
    if df.empty:
        print('no probes.parquet — skipping plot_probe_performance')
        return
    df = df[df['temp_label'] == overlay_temp].copy()
    if df.empty:
        print(f'no probes rows at temp={overlay_temp!r} — skipping plot_probe_performance')
        return
    df['group'] = [
        _probe_session_group(pt, tg)
        for pt, tg in zip(df['probe_session_type'], df['trial_tag'])
    ]
    df = df[df['group'] != 'other']

    cmap     = _model_color_map(model_paths)
    models   = [m for m in model_paths if m in set(df['model'])]
    probes   = sorted(df['probe_session_type'].unique())
    n_probes = len(probes)
    n_models = len(models)
    width    = 0.8 / max(n_models, 1)
    fig, axes = plt.subplots(1, n_probes, figsize=(max(4 * n_probes, 6), 4),
                             sharey=True, squeeze=False)
    rng = np.random.RandomState(0)
    for pi, probe in enumerate(probes):
        ax   = axes[0, pi]
        cell = df[df['probe_session_type'] == probe]
        split  = any(kw in probe for kw in _PROBE_SPLIT_KEYWORDS)
        wanted = ['training', 'probe'] if split else ['overall']
        groups_here = [g for g in wanted if (cell['group'] == g).any()]
        if not groups_here:
            ax.set_visible(False)
            continue
        x = np.arange(len(groups_here))
        for mi, model in enumerate(models):
            offset = (mi - (n_models - 1) / 2) * width
            labeled = False
            for gi, group in enumerate(groups_here):
                sub = cell[(cell['model'] == model) & (cell['group'] == group)]
                if sub.empty:
                    continue
                pooled = float(sub['dt_correct'].sum()) / float(len(sub))
                ax.bar(x[gi] + offset, pooled, width,
                       color=cmap[model], edgecolor='black', alpha=0.85, zorder=2,
                       label=(model if not labeled else None))
                labeled = True
                ps = (sub.groupby(['unit_label', 'seed'])['dt_correct']
                      .agg(['sum', 'count']).reset_index())
                ps['rate'] = ps['sum'] / ps['count']
                jitter = rng.uniform(-width / 4, width / 4, size=len(ps))
                ax.scatter(np.full(len(ps), x[gi] + offset) + jitter, ps['rate'].values,
                           s=22, alpha=0.75, color='black', edgecolors='white',
                           linewidth=0.5, zorder=3)
        ax.set_xticks(x); ax.set_xticklabels(groups_here, fontsize=9)
        ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3, axis='y')
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.set_title(probe, fontsize=10)
        if pi == 0:
            ax.set_ylabel(f'% correct (of total trials)\n@ temp={overlay_temp}')
        if pi == n_probes - 1:
            ax.legend(loc='upper right', fontsize=8, title='model')
    fig.suptitle(f'Probe performance by model — pooled correct/total @ temp={overlay_temp}; dots = subject×seed')
    fig.tight_layout()
    _save_show(fig, 'probe_performance.png', show, save_dir)


def _plot_performance_profiles_overlay(model_paths, *, overlay_temp='det',
                                       show=True, save_dir=None, reps=2000):
    frames = []
    for label, root in model_paths.items():
        ps = _per_seed_probe_scores(Path(root))
        if ps.empty:
            print(f'  note: no probes.parquet under "{label}" — dropping from performance profiles')
            continue
        ps = ps.copy(); ps['model'] = label
        frames.append(ps)
    if not frames:
        print('no probes.parquet — skipping plot_performance_profiles')
        return
    per_seed = pd.concat(frames, ignore_index=True)
    per_seed = per_seed[per_seed['temp_label'] == overlay_temp]
    if per_seed.empty:
        print(f'no probes rows at temp={overlay_temp!r} — skipping plot_performance_profiles')
        return

    cmap   = _model_color_map(model_paths)
    models = [m for m in model_paths if m in set(per_seed['model'])]
    tau    = np.linspace(0.0, 1.0, 51)
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    for model in models:
        sub = per_seed[per_seed['model'] == model].copy()
        sub['unit_seed'] = list(zip(sub['unit_label'], sub['seed']))
        mat = sub.pivot_table(index='unit_seed',
                              columns=['probe_session_type', 'trial_tag'],
                              values='rate', aggfunc='mean')
        pt, ci = _perf_profile(mat.to_numpy(), tau, reps=reps)
        color = cmap[model]
        ax.plot(tau, pt, color=color, label=model, lw=2)
        ax.fill_between(tau, ci[0], ci[1], color=color, alpha=0.25)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
    ax.set_xlabel(r'Success rate threshold $\tau$')
    ax.set_ylabel(r'P(score $\geq\tau$)')
    ax.legend(loc='upper right', fontsize=9)
    fig.suptitle(f'Performance profiles by model @ temp={overlay_temp} (bootstrap CI, reps={reps})')
    fig.tight_layout()
    _save_show(fig, 'performance_profiles.png', show, save_dir)


# ── public dispatchers ───────────────────────────────────────────────────────
def plot_training_curves(model_paths, *, show=True, save_dir=None):
    if len(model_paths) == 1:
        (root,) = model_paths.values()
        _plot_training_curves_single(Path(root), show=show, save_dir=save_dir)
    else:
        _plot_training_curves_overlay(model_paths, show=show, save_dir=save_dir)


def plot_acq_sanity(model_paths, *, show=True, save_dir=None,
                    acq_inference_trials=32, acq_pass_trials=24):
    if len(model_paths) == 1:
        (root,) = model_paths.values()
        _plot_acq_sanity_single(Path(root), show=show, save_dir=save_dir,
                                acq_inference_trials=acq_inference_trials,
                                acq_pass_trials=acq_pass_trials)
    else:
        _plot_acq_sanity_overlay(model_paths, show=show, save_dir=save_dir,
                                 acq_inference_trials=acq_inference_trials,
                                 acq_pass_trials=acq_pass_trials)


def plot_probe_performance(model_paths, *, overlay_temp='det', show=True, save_dir=None):
    if len(model_paths) == 1:
        (root,) = model_paths.values()
        _plot_probe_performance_single(Path(root), show=show, save_dir=save_dir)
    else:
        _plot_probe_performance_overlay(model_paths, overlay_temp=overlay_temp,
                                        show=show, save_dir=save_dir)


def plot_performance_profiles(model_paths, *, overlay_temp='det', show=True, save_dir=None, reps=2000):
    if len(model_paths) == 1:
        (root,) = model_paths.values()
        _plot_performance_profiles_single(Path(root), show=show, save_dir=save_dir, reps=reps)
    else:
        _plot_performance_profiles_overlay(model_paths, overlay_temp=overlay_temp,
                                           show=show, save_dir=save_dir, reps=reps)


## 5. Render

In [ ]:
# ── Render all four plots ────────────────────────────────────────────────────
# 1 model  -> section-12 layout (faceted by subject / temperature).
# >1 model -> overlay-pooled comparison (one color per model).
plot_training_curves(model_paths, show=True, save_dir=SAVE_DIR)

plot_acq_sanity(model_paths, show=True, save_dir=SAVE_DIR,
                acq_inference_trials=ACQ_INFERENCE_TRIALS,
                acq_pass_trials=ACQ_PASS_COMPLETED_TRIALS)

plot_probe_performance(model_paths, overlay_temp=OVERLAY_TEMP, show=True, save_dir=SAVE_DIR)

plot_performance_profiles(model_paths, overlay_temp=OVERLAY_TEMP, show=True, save_dir=SAVE_DIR)
